In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

#find the project root
ROOT=Path.cwd()
while not (ROOT / ".git").exists() and ROOT!=ROOT.parent:
    ROOT=ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from config import START_YEAR, END_YEAR, HOLDOUT_START, MP_THRESHOLD
import utils

df_tot=utils.player_stat_dataframe(START_YEAR, END_YEAR)
df_tot.shape

(24992, 36)

In [2]:
df_holdout=df_tot[df_tot['Year'] >= HOLDOUT_START].copy() #locked away until the very end
df_tot=df_tot[df_tot['Year'] < HOLDOUT_START].copy() #everything below uses only this
print(df_tot['Year'].max(), "last development season") #should be 2018

2018 last development season


In [3]:
#1 on every row of a player traded mid-season (his 2TM/3TM total row and each team row)
is_total=df_tot['Team'].str.endswith('TM')
df_tot['TradedMidSeason']=is_total.groupby([df_tot['Year'], df_tot['Code']]).transform('any').astype(int)

#season length = most games any single-team player played that season (50 in 1998-99, 66 in 2011-12, ...)
season_games=df_tot[~is_total].groupby('Year')['G'].max()
df_tot['SeasonG']=df_tot['Year'].map(season_games)
df_tot['GP%']=df_tot['G'] / df_tot['SeasonG']      #replaces the loader's G/82 version

#keep one row per player-season: single-team players keep their row, traded players only keep their total row
season_rows=df_tot[(df_tot['TradedMidSeason']== 0) | is_total]

#65-game rule only from 2023-24 (Year 2023) on, when it started; earlier seasons just need 20+ games
RULE_START=2023
before_rule=season_rows['Year'] < RULE_START
eligible=(before_rule & (season_rows['G'] >= 20)) | (~before_rule & (season_rows['G'] >= 65))

data=season_rows[(season_rows['MP'] >= MP_THRESHOLD) & eligible].reset_index(drop=True)

print(data.shape)
print(data.duplicated(['Year', 'Code']).sum(), "duplicate player-seasons")
print((data['MVPVS'] > 0).sum(), "vote-getter rows")
print(data.groupby('Year')['MVP'].sum().eq(1).all(), "every season still has its MVP")

(10293, 38)
0 duplicate player-seasons
648 vote-getter rows
True every season still has its MVP


In [4]:
#now feature lists and within season z-scores
feature_sets = {
    'heatmap_8':        ['PTS', 'W%', 'FT', 'eFG%', 'TRB', 'AST', 'BLK', 'STL'],
    'candidates_10':    ['PTS', 'W%', 'FT', 'eFG%', 'DRB', 'AST', 'BLK', 'STL', 'GS', '3P'],
    'rf_recall_6':      ['PTS', 'W%', 'eFG%', 'DRB', 'AST', 'BLK'],
}
#rf_recall_6 comes from the model_exploration nb which scored each subset on 2019-2025, so the results from this may be slightly more optimistic compared to the other 2
all_feats=sorted(set().union(*feature_sets.values()))

#within-season z-score for each feature: (value - season mean) / season std, then check no NaNs
for col in all_feats:
    g=data.groupby('Year')[col]
    data[col + '_z']=(data[col] - g.transform('mean'))/g.transform('std')

print(data[[c + '_z' for c in all_feats]].isna().sum())

3P_z      0
AST_z     0
BLK_z     0
DRB_z     0
FT_z      0
GS_z      0
PTS_z     0
STL_z     0
TRB_z     0
W%_z      0
eFG%_z    0
dtype: int64


### Quasi-binomial regression for MVP vote share

For each player-season, the model computes a linear score from the within-season z-scored features, then converts it to a predicted vote share between 0 and 1:
$$\text{score} = b_0 + b_1\,\text{PTS}_z + b_2\,\text{W\%}_z + \dots + b_k\,\text{STL}_z$$

$$\text{vote share} = \frac{1}{1 + e^{-\text{score}}}$$

- $b_0$ is the intercept: the score of a player at the season average in every feature.
- Each $b_i$ shows how much one standard deviation in that feature changes the score; $e^{b_i}$ is the multiplier on the odds.
- `scale='X2'` makes it quasi-binomial: the coefficients are the same as a binomial fit, but the dispersion is estimated from the data, so the standard errors and p-values are realistic.

In [5]:
fits={}
for name, feats in feature_sets.items():
    zf=[c + '_z' for c in feats]
    X=sm.add_constant(data[zf])
    fits[name]=sm.GLM(data['MVPVS'], X, family=sm.families.Binomial()).fit(scale='X2')  #quasi-binomial
    print(f"\n===== {name} (dispersion {fits[name].scale:.4f})")
    print(fits[name].summary())


===== heatmap_8 (dispersion 0.0205)
                 Generalized Linear Model Regression Results                  
Dep. Variable:                  MVPVS   No. Observations:                10293
Model:                            GLM   Df Residuals:                    10284
Model Family:                Binomial   Df Model:                            8
Link Function:                  Logit   Scale:                        0.020530
Method:                          IRLS   Log-Likelihood:                -162.35
Date:                Sun, 08 Nov 2026   Deviance:                       130.42
Time:                        17:10:43   Pearson chi2:                     211.
No. Iterations:                    13   Pseudo R-squ. (CS):            0.06855
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -1

In [6]:
def evaluate(df, score_col):
    hits, ranks=0, []
    for yr,g in df.groupby('Year'):
        assert g['MVP'].sum() == 1, f"{yr}: expected exactly one MVP, found {g['MVP'].sum()}"
        actual=g.loc[g['MVP'].idxmax(), 'Code']
        order=g.sort_values(score_col, ascending=False)['Code'].tolist()
        hits+=order[0] == actual #True gets 1
        ranks.append(order.index(actual) +1) #so top position is 1 and not 0
    print(f"{score_col}: picked MVP in {hits}/{len(ranks)} seasons,", f"real winner's avg rank={np.mean(ranks):.2f}")


In [7]:
import warnings
warnings.filterwarnings('ignore')   #hides repeated warnings from the fits

MIN_HISTORY=10 #seasons of training data before any predictions
years=sorted(data['Year'].unique())
test_years=years[MIN_HISTORY:] #1990 onward

for name, feats in feature_sets.items():
    zf=[c + '_z' for c in feats]
    col=f'pred_{name}'
    data[col]=np.nan

    for yr in test_years:
        train=data['Year'] < yr #only seasons before the test season
        test=data['Year'] == yr
        assert data.loc[train, 'Year'].max() < yr
        model=sm.GLM(data.loc[train, 'MVPVS'], sm.add_constant(data.loc[train, zf]),
                     family=sm.families.Binomial()).fit(scale='X2')
        data.loc[test, col]=model.predict(sm.add_constant(data.loc[test, zf], has_constant='add'))

tested=data[data['Year'].isin(test_years)]    #runs once, after all fits
for name in feature_sets:
    evaluate(tested, f'pred_{name}')
evaluate(tested, 'PTS')    #baseline, once

pred_heatmap_8: picked MVP in 17/29 seasons, real winner's avg rank=2.00
pred_candidates_10: picked MVP in 16/29 seasons, real winner's avg rank=1.93
pred_rf_recall_6: picked MVP in 18/29 seasons, real winner's avg rank=1.90
PTS: picked MVP in 10/29 seasons, real winner's avg rank=6.31


In [8]:
from sklearn.ensemble import HistGradientBoostingRegressor

for name, feats in feature_sets.items():
    for kind, cols in [('raw', feats), ('z', [c + '_z' for c in feats])]:
        col=f'gbr_{kind}_{name}'
        data[col]=np.nan
        for yr in test_years:
            train=data['Year'] < yr
            test=data['Year'] == yr
            assert data.loc[train, 'Year'].max() < yr
            model=HistGradientBoostingRegressor(random_state=12)
            model.fit(data.loc[train, cols], data.loc[train, 'MVPVS'])
            data.loc[test, col]=model.predict(data.loc[test, cols])

tested=data[data['Year'].isin(test_years)]      #rebuild so it includes the new gbr_ columns
for col in [c for c in data.columns if c.startswith(('pred_', 'gbr_'))]:
    evaluate(tested, col)


pred_heatmap_8: picked MVP in 17/29 seasons, real winner's avg rank=2.00
pred_candidates_10: picked MVP in 16/29 seasons, real winner's avg rank=1.93
pred_rf_recall_6: picked MVP in 18/29 seasons, real winner's avg rank=1.90
gbr_raw_heatmap_8: picked MVP in 21/29 seasons, real winner's avg rank=1.90
gbr_z_heatmap_8: picked MVP in 21/29 seasons, real winner's avg rank=1.72
gbr_raw_candidates_10: picked MVP in 22/29 seasons, real winner's avg rank=1.86
gbr_z_candidates_10: picked MVP in 20/29 seasons, real winner's avg rank=1.86
gbr_raw_rf_recall_6: picked MVP in 20/29 seasons, real winner's avg rank=1.93
gbr_z_rf_recall_6: picked MVP in 20/29 seasons, real winner's avg rank=1.86


In [9]:
#every MVP in the development seasons, as (season, player ID) pairs
mvp_rows=df_tot[df_tot['MVP']==1]
winners=set(zip(mvp_rows['Year'], mvp_rows['Code']))

#recency-weighted past MVPs: a win last season counts 1, two seasons ago 0.5, three seasons ago 0.25, and so on
from collections import defaultdict
wins=defaultdict(list)
for y, c in winners:
    wins[c].append(y)

data['MVP_recency']=[sum(0.5 ** (y - w - 1) for w in wins[c] if w < y)
                     for y, c in zip(data['Year'], data['Code'])]

#check: Jordan's values in 1992 and 1996
data.loc[data['Player']=='Michael Jordan', ['Year', 'MVP_recency', 'MVPVS', 'MVP']]

,Year,MVP_recency,MVPVS,MVP
842,1984,0.000000,0.172,0
1270,1986,0.000000,0.576,0
1475,1987,0.000000,0.831,1
1686,1988,1.000000,0.704,0
1910,1989,0.500000,0.613,0
2136,1990,0.250000,0.928,1
2370,1991,1.125000,0.938,1
2614,1992,1.562500,0.577,0
3359,1995,0.195312,0.986,1
3621,1996,1.097656,0.832,0


In [10]:
past=['MVP_recency'] #kept raw, not z-scored

for name, feats in feature_sets.items():
    q_cols=[c + '_z' for c in feats] + past
    g_cols=feats + past
    data[f'pred_past_{name}']=np.nan
    data[f'gbr_past_{name}']=np.nan

    for yr in test_years:
        train=data['Year'] < yr
        test=data['Year'] == yr
        assert data.loc[train, 'Year'].max() < yr
        qb=sm.GLM(data.loc[train, 'MVPVS'], sm.add_constant(data.loc[train, q_cols]),
                  family=sm.families.Binomial()).fit(scale='X2')
        data.loc[test, f'pred_past_{name}']=qb.predict(sm.add_constant(data.loc[test, q_cols], has_constant='add'))
        gb=HistGradientBoostingRegressor(random_state=12).fit(data.loc[train, g_cols], data.loc[train, 'MVPVS'])
        data.loc[test, f'gbr_past_{name}']=gb.predict(data.loc[test, g_cols])

tested=data[data['Year'].isin(test_years)]
for name in feature_sets:
    for col in [f'pred_{name}', f'pred_past_{name}', f'gbr_raw_{name}', f'gbr_past_{name}']:
        evaluate(tested, col)
    print()

pred_heatmap_8: picked MVP in 17/29 seasons, real winner's avg rank=2.00
pred_past_heatmap_8: picked MVP in 17/29 seasons, real winner's avg rank=2.00
gbr_raw_heatmap_8: picked MVP in 21/29 seasons, real winner's avg rank=1.90
gbr_past_heatmap_8: picked MVP in 19/29 seasons, real winner's avg rank=2.00

pred_candidates_10: picked MVP in 16/29 seasons, real winner's avg rank=1.93
pred_past_candidates_10: picked MVP in 17/29 seasons, real winner's avg rank=1.93
gbr_raw_candidates_10: picked MVP in 22/29 seasons, real winner's avg rank=1.86
gbr_past_candidates_10: picked MVP in 18/29 seasons, real winner's avg rank=1.90

pred_rf_recall_6: picked MVP in 18/29 seasons, real winner's avg rank=1.90
pred_past_rf_recall_6: picked MVP in 17/29 seasons, real winner's avg rank=1.93
gbr_raw_rf_recall_6: picked MVP in 20/29 seasons, real winner's avg rank=1.93
gbr_past_rf_recall_6: picked MVP in 17/29 seasons, real winner's avg rank=2.10



In [11]:
qb_all=sm.GLM(data['MVPVS'], sm.add_constant(data[[c + '_z' for c in feature_sets['candidates_10']] + ['MVP_recency']]),
              family=sm.families.Binomial()).fit(scale='X2')
print(qb_all.summary().tables[1])

                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const         -11.3643      0.127    -89.420      0.000     -11.613     -11.115
PTS_z           1.5146      0.040     37.404      0.000       1.435       1.594
W%_z            1.8546      0.036     51.501      0.000       1.784       1.925
FT_z            0.2254      0.028      8.031      0.000       0.170       0.280
eFG%_z          0.3936      0.025     15.465      0.000       0.344       0.444
DRB_z           0.4303      0.021     20.227      0.000       0.389       0.472
AST_z           0.6302      0.019     33.205      0.000       0.593       0.667
BLK_z           0.2166      0.019     11.170      0.000       0.179       0.255
STL_z           0.2367      0.020     12.041      0.000       0.198       0.275
GS_z            1.1088      0.075     14.863      0.000       0.963       1.255
3P_z           -0.1368      0.019     -7

In [12]:
# print(qb_all.pvalues['MVP_recency'])

from the last line in the table--> MVP_recency = 0.337, p < 0.001 → e^0.337 ≈ 1.40, so a reigning MVP (with recency=1) has about 40% higher odds of vote share than a player with identical stats and no recent MVP. It doesn't improve picking the winner in forward-chaining.

In [13]:
from statsmodels.discrete.conditional_models import ConditionalLogit

#conditional logit--> within each season, which player wins? (MVP = 1 for the winner, 0 otherwise)
clogit_cols=[c + '_z' for c in feature_sets['candidates_10']] + ['MVP_recency']
clogit=ConditionalLogit(data['MVP'], data[clogit_cols], groups=data['Year']).fit()
print(clogit.summary())

                  Conditional Logit Model Regression Results                  
Dep. Variable:                    MVP   No. Observations:                10293
Model:               ConditionalLogit   No. groups:                         39
Log-Likelihood:               -27.458   Min group size:                    199
Method:                          BFGS   Max group size:                    329
Date:                Sun, 08 Nov 2026   Mean group size:                 263.9
Time:                        17:13:12                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
PTS_z           3.9206      1.121      3.498      0.000       1.724       6.118
W%_z            5.3840      1.201      4.484      0.000       3.031       7.737
FT_z           -0.0584      0.385     -0.152      0.879      -0.813       0.696
eFG%_z         -0.1333      0.455     -0.293   

In the conditional logit (who wins)--> MVP_recency = −0.81 (p = 0.18) → e^−0.81 ≈ 0.45: recent MVPs look about 55% less likely to win at the same stats (voter fatigue), but with only 39 seasons this isn't statistically significant.

In [14]:
#forward-chaining conditional logit: for each test season, train only on earlier seasons
def forward_clogit(df, cols, col_name):
    df[col_name]=np.nan
    biggest={}     #largest coefficient per test season, to check stability
    for yr in test_years:
        train=df['Year'] < yr
        test=df['Year'] == yr
        assert df.loc[train, 'Year'].max() < yr
        res=ConditionalLogit(df.loc[train, 'MVP'], df.loc[train, cols],
                             groups=df.loc[train, 'Year']).fit(disp=False, skip_hessian=True, maxiter=200)
        df.loc[test, col_name]=df.loc[test, cols] @ res.params     #score = sum of weight x feature
        biggest[yr]=res.params.abs().max()
    return pd.Series(biggest)

base_cols=[c + '_z' for c in feature_sets['candidates_10']]
stab=forward_clogit(data, base_cols, 'clogit_candidates_10')
stab_past=forward_clogit(data, base_cols + ['MVP_recency'], 'clogit_past_candidates_10')

print(stab.round(1).to_string())

1990     42.2
1991     88.8
1992     91.4
1993     86.2
1994    187.1
1995     69.5
1996     69.8
1997     70.4
1998     11.3
1999     11.7
2000     11.6
2001      5.9
2002      6.1
2003      6.5
2004      6.7
2005      6.5
2006      5.5
2007      5.6
2008      5.7
2009      5.9
2010      5.5
2011      6.2
2012      5.4
2013      5.3
2014      5.2
2015      5.6
2016      5.6
2017      5.6
2018      5.6


In [15]:
#conditional logit fits are unstable before 2001 (fewer than ~20 winners to learn from--> see the stability numbers above), so I'm also comparing on 2001-2018

#compare all models: all test seasons (1990-2018) and the seasons where the conditional logit is stable (2001-2018)
compare=['clogit_candidates_10', 'clogit_past_candidates_10',
         'pred_candidates_10', 'gbr_raw_candidates_10', 'gbr_z_candidates_10', 'PTS']

for label, start in [('1990-2018', 1990), ('2001-2018 (stable conditional logit)', 2001)]:
    print(f"\n===== {label}")
    window=data[(data['Year'] >= start) & data['Year'].isin(test_years)]
    for col in compare:
        evaluate(window, col)


===== 1990-2018
clogit_candidates_10: picked MVP in 18/29 seasons, real winner's avg rank=1.86
clogit_past_candidates_10: picked MVP in 16/29 seasons, real winner's avg rank=1.90
pred_candidates_10: picked MVP in 16/29 seasons, real winner's avg rank=1.93
gbr_raw_candidates_10: picked MVP in 22/29 seasons, real winner's avg rank=1.86
gbr_z_candidates_10: picked MVP in 20/29 seasons, real winner's avg rank=1.86
PTS: picked MVP in 10/29 seasons, real winner's avg rank=6.31

===== 2001-2018 (stable conditional logit)
clogit_candidates_10: picked MVP in 13/18 seasons, real winner's avg rank=1.83
clogit_past_candidates_10: picked MVP in 12/18 seasons, real winner's avg rank=1.89
pred_candidates_10: picked MVP in 10/18 seasons, real winner's avg rank=2.17
gbr_raw_candidates_10: picked MVP in 13/18 seasons, real winner's avg rank=2.28
gbr_z_candidates_10: picked MVP in 11/18 seasons, real winner's avg rank=2.28
PTS: picked MVP in 4/18 seasons, real winner's avg rank=8.89
